# CNN seizure prediction on CHB-MIT — walkthrough

**Question:** does the graph in the ST-GNN actually help? This notebook replaces the ST-GNN window
encoder with two plain CNNs and keeps everything else identical (data, labels, leakage-free splits,
5-minute context model, seizure-level scoring), so any difference comes from the encoder alone.

| Encoder | Input | Idea |
|---|---|---|
| **EEGNet** (Lawhern et al., 2018) | raw 5-s window, 18 channels × 640 samples | learns frequency filters, then spatial filters across channels |
| **Spectrogram CNN** (after Truong et al., 2018) | 0–40 Hz log-spectrogram of each channel | 2-D CNN over frequency × time, channels as input planes |

### How to use this notebook
Run the cells **top to bottom** (Shift+Enter). Sections 1–6 take about 20–30 minutes on the RTX 5090
and train one fold you can watch. Section 7 starts the full experiments in the background (several
hours); section 8 collects the results whenever they are ready, and you can re-run it at any time.

### What is in this repository

| Path | What it is | Do you need to touch it? |
|---|---|---|
| `CNN_walkthrough.ipynb` | **this notebook — start here** | yes |
| `src/` | the code the notebook calls (models, training, evaluation) | only to read it; the notebook shows the important parts |
| `tests/` | automatic checks of the code (`python -m pytest -q tests`) | no |
| `run_all.sh` | the same experiments as section 7, as one terminal command | optional |
| `data/` | link to the preprocessed CHB-MIT windows from the ST-GNN repo (not stored in git) | no |
| `results/`, `logs/` | created by the runs (not stored in git) | read only |

In `src/`, the files that are new for the CNNs are `cnn.py` (the two models), `train.py`
(training loop) and `context.py` (5-minute context model). Everything else (`preprocess.py`,
`data.py`, `splits.py`, `metrics.py`, `evaluate.py`, ...) is the tested pipeline from the ST-GNN repo, unchanged.

## 0. Setup

In [ ]:
import os, sys, json, time, subprocess, inspect
from pathlib import Path
from types import SimpleNamespace

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython.display import Code, Image, Markdown, display

# Preprocessed data: this repo's data/ link, or the ST-GNN repo next to it
CANDIDATES = [ROOT / "data/processed_v3", Path.home() / "seizure_stgnn_v3/data/processed_v3"]
PROCESSED = next((p for p in CANDIDATES if p.exists()), None)
assert PROCESSED is not None, "No preprocessed data found: run src.preprocess first (see README)"
if not (ROOT / "data").exists():
    (ROOT / "data").symlink_to(PROCESSED.parent)          # so the scripts find data/processed_v3 too
STGNN_RESULTS = Path.home() / "seizure_stgnn_v3/results"   # for the comparison in section 8

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = torch.bfloat16 if DEVICE.type == "cuda" else None
print("repo      :", ROOT)
print("data      :", PROCESSED)
print("torch     :", torch.__version__, "| device:", DEVICE,
      f"({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else "")

def show(obj):
    "Display the source code of a function or class from src/."
    display(Code(inspect.getsource(obj), language="python"))

## 1. The data

Each patient's recordings were already cut into 5-second windows (18 bipolar channels, 128 Hz,
0.5–40 Hz band-pass) by `src/preprocess.py` in the ST-GNN repo. Every window has a label:

* **preictal** — within 30 minutes before a seizure (what we want to detect),
* **interictal** — at least 60 minutes away from any seizure (normal),
* everything else (seizures, the 5 minutes after them, the 30–60-minute buffer) is not used.

Windows are kept in recording order with their real timestamps, so the evaluation can count
seizures and false alarms per hour of recording rather than individual windows.

In [ ]:
from src.data import list_subjects, load_subject

SUBJECTS = list_subjects(PROCESSED)
DATA = {s: load_subject(PROCESSED, s) for s in SUBJECTS}      # memory-mapped: nothing large is loaded yet

print(f"{'patient':8s} {'windows':>8s} {'seizures':>9s} {'interictal h':>13s} {'preictal h':>11s}")
for s, d in DATA.items():
    print(f"{s:8s} {len(d):8d} {d.n_seizures:9d} {np.sum(d.hard == 0) * 5 / 3600:13.1f} {np.sum(d.hard == 1) * 5 / 3600:11.1f}")
tot = sum(len(d) for d in DATA.values())
print(f"\n{len(SUBJECTS)} patients, {tot:,} windows, {sum(d.n_seizures for d in DATA.values())} seizures with a usable preictal period")

### What one window looks like: normal vs. 2 minutes before a seizure

In [ ]:
P = "chb01"
d = DATA[P]
onset_idx = np.where(d.block >= 0)[0]
i_pre = onset_idx[np.argmax(d.risk[onset_idx])] - 24           # ~2 minutes before the first onset
i_int = np.where(d.hard == 0)[0][len(np.where(d.hard == 0)[0]) // 2]
ch = d.meta["channels"]

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
for ax, i, title in ((axes[0], i_int, "interictal (normal)"), (axes[1], i_pre, "preictal (~2 min before onset)")):
    x = np.asarray(d.window(i)[0], float)
    x = (x - x.mean(1, keepdims=True)) / (x.std(1, keepdims=True) + 1e-6)   # the same z-score the models see
    t = np.arange(x.shape[1]) / 128
    for c in range(18):
        ax.plot(t, x[c] * 0.3 - c, lw=0.5, color="0.2")
    ax.set_title(f"{P}, {title}", loc="left"); ax.set_xlabel("seconds")
axes[0].set_yticks(-np.arange(18)); axes[0].set_yticklabels(ch, fontsize=7)
plt.tight_layout(); plt.show()
print("Hard to tell apart by eye, which is why seizure prediction is hard.")

### The same windows as the spectrogram CNN sees them
`LogSpectrogram` (in `src/cnn.py`) turns each channel into log power over 0–40 Hz (1-Hz bins) and
21 time steps (1-s windows every 0.25 s). The CNN gets an 18 × 41 × 21 "image" per window.

In [ ]:
from src.cnn import LogSpectrogram, zscore
spec = LogSpectrogram()
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, i, title in ((axes[0], i_int, "interictal"), (axes[1], i_pre, "preictal")):
    x = zscore(torch.as_tensor(np.asarray(d.window(i)[0], np.float32))[None])
    s = spec(x)[0].mean(0).numpy()                                 # averaged over the 18 channels
    im = ax.imshow(s, origin="lower", aspect="auto", extent=[0, 5, 0, 40], cmap="magma")
    ax.set_title(f"{P}, {title} (mean over channels)", loc="left"); ax.set_xlabel("seconds"); ax.set_ylabel("Hz")
plt.colorbar(im, ax=axes, shrink=0.8, label="standardised log power"); plt.show()

## 2. The two CNN encoders

Both models output a **64-number embedding** per window (`embed`) and a **risk logit** (`forward`).
The embedding is what the 5-minute context model reads later, exactly as with the ST-GNN.

In [ ]:
from src.cnn import EEGNet, SpectroCNN, build, count_parameters
show(EEGNet)

In [ ]:
show(SpectroCNN)

In [ ]:
x = torch.randn(8, 18, 640)
for arch in ("eegnet", "spectro"):
    m = build(arch).eval()
    with torch.no_grad():
        print(f"{arch:8s} parameters: {count_parameters(m):>8,}   embed: {tuple(m.embed(x).shape)}   logit: {tuple(m(x).shape)}")
print("(the ST-GNN with band features has about 0.6 M parameters)")

## 3. How a model is trained

* **Balanced epochs.** Only ~3% of windows are preictal, so each epoch draws 40,000 windows, half
  preictal and half interictal.
* **Loss.** `0.5 × MSE(risk) + 0.5 × BCE(label)`. The soft risk rises from 0.1 at 30 min before onset
  to 1.0 at onset, so windows close to a seizure count more.
* **Model selection.** After each epoch the model is scored (window AUC) on the **validation
  patients only**; the best epoch is kept and training stops after 8 epochs without improvement.
  The test patient is never used until the end.
* **Augmentation** (optional): random time shift, channel gain, noise, time masking and channel
  dropout, applied on the GPU.

In [ ]:
from src import train as T
show(T.train_fold)

## 4. Leakage-free splits

**Leave-one-patient-out (LOPO):** one fold per patient. The model trains on 20 patients, chooses its
best epoch and alarm threshold on 3 others, and is tested on the one patient it has never seen.

**Chronological (patient-specific):** within one patient, train on the first ~50% of the recording,
validate on the next 20%, test on the last 30%. Only patients with at least 3 seizures qualify.

In [ ]:
from src.splits import lopo_folds, chronological_split
FOLDS = {f.name: f for f in lopo_folds(SUBJECTS, {s: d.n_seizures for s, d in DATA.items()}, n_val=3, seed=42)}
for name in list(FOLDS)[:5]:
    f = FOLDS[name]
    print(f"fold {name}: test {f.test}  val {f.val}  train {len(f.train)} patients")
print(f"... {len(FOLDS)} folds in total")
print("chronological folds:", [s for s in SUBJECTS if chronological_split(DATA[s].block) is not None])

## 5. Train one fold and watch it

This trains EEGNet for the fold whose **test patient is chb01** (about 5–10 minutes), then the
spectrogram CNN on the same fold. Each line is one epoch; `*` marks a new best validation AUC.
Change `DEMO` to try another patient.

In [ ]:
DEMO = "chb01"
fold = FOLDS[DEMO]
tr, va, te = ([DATA[s] for s in grp] for grp in (fold.train, fold.val, fold.test))

def demo_args(arch, **kw):
    a = dict(arch=arch, protocol="lopo", epochs=30, patience=8, samples_per_epoch=40000, batch_size=256,
             lr=1e-3, weight_decay=1e-3, alpha=0.5, augment=False, n_val=3, val_max_interictal=20000,
             num_workers=4, seed=42)
    a.update(kw); return SimpleNamespace(**a)

DEMO_RUNS = {}
for arch in ("eegnet", "spectro"):
    args = demo_args(arch)
    print(f"\n=== {arch} | test {fold.test} | val {fold.val}")
    t0 = time.time()
    model, hist, best = T.train_fold(tr, va, args, DEVICE, AMP)
    out = ROOT / f"results/demo_{arch}"
    T.save_fold(str(out), DEMO, model, hist, best, va, te, args, DEVICE, AMP, fold.train)
    DEMO_RUNS[arch] = (out, hist)
    print(f"done in {(time.time() - t0) / 60:.1f} min, best validation AUC {best:.3f}")

In [ ]:
plt.figure(figsize=(7, 3.5))
for arch, (_, hist) in DEMO_RUNS.items():
    plt.plot([h["epoch"] for h in hist], [h["val_auc"] for h in hist], marker="o", ms=3, label=arch)
plt.xlabel("epoch"); plt.ylabel("validation AUC (other patients)"); plt.legend(frameon=False)
plt.title(f"fold {DEMO}: learning curves", loc="left"); plt.show()

## 6. From window scores to seizure alarms

A risk score per window is not yet a prediction system. `src/evaluate.py`:

1. picks the alarm threshold τ on the **validation patients** (best seizure sensitivity with at
   most 0.5 false alarms per hour),
2. raises an alarm when 3 of the last 5 windows exceed τ, then stays quiet for 30 minutes,
3. counts a seizure as **predicted** if an alarm fires in its 30-minute preictal period, and every
   other alarm as a **false alarm**,
4. compares with a **random predictor** that raises alarms at the same rate (p-value).

In [ ]:
from src import evaluate as E, figures as F

for arch, (out, _) in DEMO_RUNS.items():
    s = E.evaluate(str(out), lead_gap_h=0)
    display(Markdown(f"#### {arch}\n" + open(out / "results.md").read().split("| Subject")[0]))

    preds = E.load_predictions(str(out / DEMO / "predictions.npz"))
    v = preds["test"][DEMO]
    tau = json.load(open(out / "results.json"))["subjects"][0]["tau"]
    png = out / f"risk_{DEMO}.png"
    F.risk_timeline(v, DEMO, tau, str(png))
    display(Image(str(png)))

### Add the 5-minute context model
A small GRU reads the last 60 window embeddings (5 minutes) and outputs the risk for the current
window. In the ST-GNN repo this step gave the biggest improvement.

> **Correction:** this context model (`src/context.py`) builds its history from labelled windows only,
> which leaks the label (see `Long_context_walkthrough.ipynb`). Its numbers here and in sections 8–9 are
> overstated; the corrected model is `src/long_context.py`.

In [ ]:
from src import context as Cx
show(Cx.ContextGRU)
for arch, (out, _) in DEMO_RUNS.items():
    ctx_out = ROOT / f"results/demo_context_{arch}"
    Cx.main(["--processed_dir", str(PROCESSED), "--source", str(out), "--out_dir", str(ctx_out)])
    E.evaluate(str(ctx_out))
    display(Markdown(f"#### context GRU on {arch}\n" + open(ctx_out / "results.md").read().split("| Subject")[0]))

**One fold is one patient**, so these numbers vary a lot from patient to patient. The real
comparison needs all 24 folds, which is what section 7 runs.

## 7. Full experiments (background, about 75 minutes on the RTX 5090)

The next cell writes `logs/full_runs.sh` and starts it in the background, so it keeps running if
you close the notebook. Order: EEGNet and spectrogram CNN, each under LOPO then chronological,
each followed by its context model and the seizure-level evaluation (all seizures and lead seizures).

If every result already exists the cell does nothing; set `FORCE_RERUN = True` to train everything again.

In [ ]:
STEPS = []
for proto in ("lopo", "chrono"):
    for arch in ("eegnet", "spectro"):
        enc, ctx = f"{arch}_{proto}", f"context_{arch}_{proto}"
        STEPS += [f"python -m src.train --processed_dir data/processed_v3 --arch {arch} --protocol {proto} --out_dir results/{enc}",
                  f"python -m src.context --processed_dir data/processed_v3 --source results/{enc} --out_dir results/{ctx}"]
        for r in (enc, ctx):
            STEPS += [f"python -m src.evaluate --results_dir results/{r} > /dev/null",
                      f"python -m src.evaluate --results_dir results/{r} --lead_gap_h 4 --processed_dir data/processed_v3 --out_dir results/{r}/lead4h > /dev/null",
                      f"python -m src.figures --results_dir results/{r} > /dev/null"]

os.makedirs("logs", exist_ok=True)
script = ["#!/usr/bin/env bash", f"cd {ROOT}", f"export PATH={Path(sys.executable).parent}:$PATH"]
script += [f'echo "[$(date +%H:%M)] {s.split(" > ")[0]}"; {s} || echo "STEP_FAILED: {s[:60]}"' for s in STEPS]
script += ['echo "ALL_FINISHED $(date)"']
Path("logs/full_runs.sh").write_text("\n".join(script) + "\n")

FORCE_RERUN = False
FINAL = [f"results/context_{a}_{p}/lead4h/results.json" for a in ("eegnet", "spectro") for p in ("lopo", "chrono")]
running = subprocess.run("pgrep -f 'bash logs/full_[r]uns.sh'", shell=True, capture_output=True, text=True).stdout.strip()
if all(Path(f).exists() for f in FINAL) and not FORCE_RERUN:
    print("All runs already finished: nothing to do (set FORCE_RERUN = True to train again).")
elif running:
    print("Already running (pid", running, ") — not starting a second copy.")
else:
    subprocess.Popen("setsid nohup bash logs/full_runs.sh > logs/full_runs.log 2>&1 &", shell=True)   # survives closing the notebook
    print(f"Started {len(STEPS)} steps in the background. Log: logs/full_runs.log")

In [ ]:
# Progress: re-run this cell any time
log = Path("logs/full_runs.log")
if not log.exists():
    print("not started")
else:
    lines = log.read_text().splitlines()
    print("\n".join(l for l in lines if l.startswith("[")) or "starting...")
    print(f"\nlast lines:\n" + "\n".join(lines[-4:]))
    print("\nFINISHED" if any("ALL_FINISHED" in l for l in lines) else "\nstill running")

## 8. Results: CNNs vs. the ST-GNN

Collects every finished run. **All seizures** counts every seizure; **lead seizures** counts only
seizures that start at least 4 hours after the previous one. Lead seizures are the stricter test:
right after a seizure, "another one is coming" is easy to guess without reading the EEG.
The ST-GNN rows are read from the ST-GNN repo's results on this machine.

In [ ]:
def summary(path):
    p = Path(path) / "results.json"
    if not p.exists():
        return None
    return json.loads(p.read_text().replace("NaN", "null"))["summary"]

ROWS = [("ST-GNN + bands (graph)",          STGNN_RESULTS / "stgnn_bands_lopo"),
        ("EEGNet",                          ROOT / "results/eegnet_lopo"),
        ("Spectrogram CNN",                 ROOT / "results/spectro_lopo"),
        ("Context GRU on ST-GNN + bands",   STGNN_RESULTS / "context_stgnn_bands_lopo"),
        ("Context GRU on EEGNet",           ROOT / "results/context_eegnet_lopo"),
        ("Context GRU on spectrogram CNN",  ROOT / "results/context_spectro_lopo")]

def fmt_p(p):
    return "—" if p is None else ("< 1e-4" if p < 1e-4 else f"{p:.3g}")

def table(rows, title):
    out = [f"#### {title}", "",
           "| Model | All: predicted | FA/h | p | Lead: predicted | FA/h | chance | p | window AUC |",
           "|---|---|---|---|---|---|---|---|---|"]
    for name, path in rows:
        a, l = summary(path), summary(Path(path) / "lead4h")
        if a is None:
            out.append(f"| {name} | *not run yet* | | | | | | | |"); continue
        lt = (f"{l['predicted']}/{l['seizures']} | {l['fpr_per_hour']:.2f} | {l['random_predictor_sensitivity']:.0%} | "
              f"{fmt_p(l['p_value_vs_chance'])}") if l else "— | — | — | —"
        out.append(f"| {name} | {a['predicted']}/{a['seizures']} ({a['sensitivity']:.0%}) | {a['fpr_per_hour']:.2f} | "
                   f"{fmt_p(a['p_value_vs_chance'])} | {lt} | {a['window_auc_mean_per_subject']:.3f} |")
    display(Markdown("\n".join(out)))

table(ROWS, "Leave-one-patient-out (24 test patients)")
table([(n.replace("lopo", "chrono"), Path(str(p).replace("_lopo", "_chrono"))) for n, p in ROWS],
      "Patient-specific, chronological (13 patients)")

In [ ]:
# Bar chart: seizure sensitivity vs. a random predictor at the same false-alarm rate
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, sub, title in ((axes[0], "", "all seizures"), (axes[1], "lead4h", "lead seizures only")):
    names, sens, chance = [], [], []
    for name, path in ROWS:
        s = summary(Path(path) / sub)
        if s:
            names.append(name); sens.append(s["sensitivity"]); chance.append(s["random_predictor_sensitivity"])
    y = np.arange(len(names))
    ax.barh(y, sens, color="#3b6ea5", label="model")
    ax.scatter(chance, y, marker="|", s=300, color="#c0504d", label="random predictor", zorder=3)
    ax.set_yticks(y); ax.set_yticklabels(names); ax.invert_yaxis()
    ax.set_xlim(0, 1); ax.set_xlabel("seizures predicted"); ax.set_title(f"LOPO, {title}", loc="left")
axes[0].legend(frameon=False, loc="lower right"); plt.tight_layout(); plt.show()

## 9. Conclusions

> **Correction (5 Oct 2026).** Points 2 and 3 below rely on the 5-minute context model, whose history
> leaked the label. With history read from every recorded window (`Long_context_walkthrough.ipynb`), the
> context model gives ST-GNN + bands 61 ± 1/151 and EEGNet 53 ± 3/151 (5 min) or 61 ± 2/151 (60 min), and
> no context model is robustly above chance on lead seizures. Point 1 (window encoders) is unaffected.

Numbers from the tables above (leave-one-patient-out, one training seed per model).

**1. On their own, all three encoders barely generalise to a new patient.** Window AUC is 0.51–0.52
for the ST-GNN, EEGNet and the spectrogram CNN alike. Each beats a random predictor on all
seizures at best weakly (ST-GNN p = 0.001, EEGNet p = 0.05, spectrogram CNN p = 0.17), and none does
on lead seizures. Raw EEG, spectrograms and channel-synchrony graphs give the same result: whatever
precedes a seizure in one patient's 5-second windows mostly does not carry over to another patient.

**2. The 5-minute context model matters more than the encoder.** Adding the GRU improves every
encoder: ST-GNN 52 → 66 of 151 seizures, EEGNet 50 → 61, spectrogram CNN 39 → 44.

**3. With context, the graph encoder is best.** ST-GNN + bands with context predicts 66/151 seizures
at 0.37 false alarms/h and is the only model clearly above chance on lead seizures (21/65 vs. 15%
expected by chance, p = 4 × 10⁻⁴). EEGNet with context is second (61/151 at 0.56 FA/h; lead 16/65,
p = 0.18) and the spectrogram CNN third (44/151; lead 15/65, p = 0.07). Phase-synchrony features between
channels seem to give the context model a slightly more transferable signal than learned CNN
filters do. The margin is small, though (21 vs. 16 lead seizures, one seed each), so this is
suggestive, not proven.

**4. Patient-specific models are too data-limited to rank.** Trained on the first half of one
patient's recording (often 2–3 seizures), every encoder predicts 2–8 of 31 test seizures and at most
3 of 16 lead seizures. Differences there are within noise.

**What this means for the next step:** changing the window encoder does not move the needle; the
gain came from time context. The natural next experiment is a **longer context** (30–60 minutes of
window embeddings, matching the 30-minute preictal period) and repeating the best models with
several seeds to put error bars on the differences.